# Laboratory Exercise 2: Online Orders

Basket is an online shop that delivers across Macedonia and takes orders on its website, in its mobile app, and through a marketplace. Its export in `data/orders.csv` holds one row per order, and `data/new-orders.csv` counts the orders placed on each day. The orders are synthetic and describe no real customer, shop, or marketplace.

The columns of `orders.csv` are these:

- `order_id`, the identifier of the order, and `ordered_on`, the day it was placed,
- `city`, the city of delivery, and `sales_channel`, which is the website, the mobile app, or the marketplace,
- `customer_type`, a consumer or a business, and `items`, the number of items as the order system recorded it,
- `membership`, the customer's loyalty level, one of none, bronze, silver, gold, and platinum,
- `payment_method`, and `delivery`, which is home, a parcel locker, or store pickup, with `delivery_fee`, the fee in euros,
- `package_weight`, the weight of the package in kilograms, `order_value`, the value of the goods in euros, and `delivery_days`, the days until the order arrived,
- `discount`, the discount in euros, and `account_age_days`, the days since the customer opened the account, and
- `rating`, the customer's rating of the order from 1 to 5, which a later model will predict.

The orders are to be prepared for that model: every gap filled for a reason, every unusual value understood, every label turned into numbers, and every column brought to a comparable scale, with every step learned from the training orders alone. No request asks for a model.

Answer each request in the cells below it, and add as many cells as you need. The laboratory test asks for values and decisions that your notebook computes, so keep every result visible, run the notebook from top to bottom before you submit it, and make sure it runs on a machine without internet access.

## Setup

The cells in this section are written already. Run them before the tasks.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, MissingIndicator, SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer, RobustScaler,
                                   StandardScaler)

DATA = Path("data")
# Every random step uses this seed, so a rerun gives the same result.
SEED = 2026

In [ ]:
# The course style.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#1e2a24"]
GROUND, INK, GRID = "#ffffff", "#1e2a24", "#e2e6e4"
plt.rcParams.update(
    {
        "figure.figsize": (8, 4.5),
        "figure.facecolor": GROUND,
        "figure.constrained_layout.use": True,
        "savefig.facecolor": GROUND,
        "savefig.transparent": False,
        "font.family": "sans-serif",
        "font.sans-serif": ["Archivo", "DejaVu Sans"],
        "font.size": 12,
        "text.color": INK,
        "axes.facecolor": GROUND,
        "axes.edgecolor": INK,
        "axes.labelcolor": INK,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.prop_cycle": plt.cycler(color=PALETTE),
        "grid.color": GRID,
        "xtick.color": INK,
        "ytick.color": INK,
        "legend.frameon": False,
        "image.cmap": "cividis",
    }
)
sns.set_palette(PALETTE)

In [ ]:
class FrequencyEncoder(TransformerMixin, BaseEstimator):
    """Replaces each category with its share of the rows passed to fit, and an unseen category with 0."""

    def fit(self, X: pd.DataFrame, y: object = None) -> "FrequencyEncoder":
        self.shares_ = {column: X[column].value_counts(normalize=True) for column in X.columns}
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        return np.column_stack([X[column].map(share).fillna(0).to_numpy(dtype=float)
                                for column, share in self.shares_.items()])

    def get_feature_names_out(self, input_features: object = None) -> np.ndarray:
        return np.array([f"{column}_frequency" for column in self.shares_])

In [ ]:
orders = pd.read_csv(DATA / "orders.csv")
new_orders = pd.read_csv(DATA / "new-orders.csv")
orders.head()

## Task 1: The Gaps

### 1.1 Reading the Gaps

The file writes a value it does not have with a marker of its own. Read the orders into a table named `df` in which every such value is missing. Count the missing values in each column, and show where they fall.

In [ ]:
# Write your solution here.

### 1.2 Why Values Are Missing

For each column with missing values, find out whether its gaps depend on another column, whether a gap means that the value does not exist, or whether the gaps follow no pattern. Show the comparison that decides each case.

In [ ]:
# Write your solution here.

### 1.3 What Deleting Would Cost

Find out how many orders would remain if every order with a missing value were dropped, and how the mix of sales channels would change. Then decide what to do with the orders that state no rating, and apply the decision to `df`.

In [ ]:
# Write your solution here.

## Task 2: The Split and the Fills

### 2.1 The Test Orders

The orders placed on 1 September 2026 or later stand in for orders that the preparation has never seen. Set them aside as the test orders, `X_test` and `y_test`, and keep the earlier orders as the training orders, `X_train` and `y_train`, where `X` holds every column except the rating and `y` holds the rating. Every later step learns only from the training orders. Report the number of each.

In [ ]:
# Write your solution here.

### 2.2 Filling the Gaps

Fill the gaps of each column in the way its pattern calls for, learning every fill from the training orders, and keep a record of the filled values wherever the absence itself carries information. Report the value that each fill by a single value uses.

In [ ]:
# Write your solution here.

### 2.3 How Well a Fill Recovers the Weight

Hide the known package weight of every training order whose identifier is divisible by five. Fill these weights back in three ways, all from the numeric columns: with the median weight, from the five most similar orders judged on standardized columns, and by predicting each column from the others in rounds with the seed 2026. Measure how far each fill lies from the true weights on average, and show the filled weights against the true ones.

In [ ]:
# Write your solution here.

### 2.4 Gaps in Time

`new_orders` counts the orders placed on each day, and a day without a row was not recorded. Make every day of the period visible, then fill each gap from the neighbouring days, at most three days into a gap and only between recorded days. Report how many days are missing before the fill and how many stay missing after it, and show the series around its longest gap.

In [ ]:
# Write your solution here.

## Task 3: Outliers

### 3.1 Unusual Numbers of Items

Flag the unusual numbers of items among the training orders in two ways: as more than 1.5 interquartile ranges beyond the quartiles, and as more than three standard deviations from the mean. Report how many orders each way flags, show them, and explain why the two ways disagree.

In [ ]:
# Write your solution here.

### 3.2 Errors or Genuine Values

Decide for each flagged number of items whether it is an error or a genuine value, and handle it accordingly in `X`, `X_train`, and `X_test`.

In [ ]:
# Write your solution here.

### 3.3 Unusual Combinations

Among the training orders of consumers with a known number of items and a known package weight, find the orders whose combination of items and weight is unusual, in two ways: by how quickly random splits isolate an order, and by how much sparser its neighbourhood is than the neighbourhoods of its neighbours. Judge both on the two columns standardized, treat 2% of the orders as unusual, use 20 neighbours, and use the seed 2026. Decide what to do with the orders that both ways flag.

In [ ]:
# Write your solution here.

### 3.4 Capping the Delivery Days

Find the 1st and the 99th percentiles of the delivery days among the training orders, and count the test orders that lie beyond them. Decide whether to cap the delivery days at these bounds.

In [ ]:
# Write your solution here.

## Task 4: Encoding and Scaling

### 4.1 Labels as Numbers

Turn every label into numbers in the way its kind calls for, learning from the training orders:

- the membership keeps its order, from none to platinum,
- a label without an order gets one column per category, a single column when it has two categories, and zeros for a category that the training orders never held, and
- the city becomes its share of the training orders, with 0 for a city that the training orders never held.

Report the code of each membership level, the number of columns that the labels without an order become, and the share of Kumanovo.

In [ ]:
# Write your solution here.

### 4.2 Comparable Scales

Bring every numeric column to a scale that suits it, and reduce the skew of the columns with a long right tail. Report the skew of the account age before and after, the power that brings it closest to a normal distribution, and what the discount needs instead.

In [ ]:
# Write your solution here.

## Task 5: One Pipeline and Leakage

### 5.1 One Preparation

Join every step of Tasks 2 to 4 into one preparation that is fitted on the training orders and only applied to the test orders. Report the number of rows and columns of the prepared training and test tables, and confirm that they hold no missing values.

In [ ]:
# Write your solution here.

### 5.2 Fitted on All Orders

Fit the same preparation on all orders, the test orders included, and compare what it learns for the cities that only the test orders hold with what the preparation fitted on the training orders learns. Then show how the leak changes the test orders on the delivery days. Standardize the known test delivery days once with the mean and the standard deviation of the training orders and once with those of all orders, compare how far they spread, and count the test orders beyond the 99th percentile of the delivery days of the training orders and of all orders.

In [ ]:
# Write your solution here.

### 5.3 The Decisions

Record every preparation decision in a table, with the reason the data give for it.